# 输入层
sentence -> tokenizor -> padding -> embedding -> add position embedding
-> masked attention

In [2]:
import tiktoken
import torch 
import torch.nn as nn
import numpy as np 
encoding = tiktoken.get_encoding("gpt2")

In [3]:
batch = 2
max_len = 16
vocab_size = encoding.n_vocab
data = ["你好！","回家了吧。"]
print(encoding.encode(data[0]))
print(encoding.encode(data[1]))
x = [encoding.encode(sentence) for sentence in data ]
valid_length = [len(sentence) for sentence in x]
y = [sentence[1:] + [-100]*(max_len+1-len(sentence)) if max_len-len(sentence)>0 else sentence[1:]  for sentence in x ]
x = [sentence + [encoding.eot_token]*(max_len-len(sentence)) if max_len-len(sentence)>0 else sentence for sentence in x ]
x= torch.tensor(x)
y = torch.tensor(y)
pad_mask = torch.ones(batch,max_len,max_len)==0
for i,length in enumerate(valid_length):
    pad_mask[i,:,length:] = True

[19526, 254, 25001, 121, 171, 120, 223]
[32368, 252, 22522, 114, 12859, 228, 28938, 100, 16764]


In [4]:
y

tensor([[  254, 25001,   121,   171,   120,   223,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100],
        [  252, 22522,   114, 12859,   228, 28938,   100, 16764,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100]])

In [5]:
word_embedding = torch.nn.Embedding(encoding.n_vocab,256)
pos_embedding = torch.nn.Embedding(max_len,256)
x = torch.tensor(x)
embedding = word_embedding(x) + pos_embedding(torch.arange(max_len)).unsqueeze(0)

C:\Users\lifeng\AppData\Local\Temp\ipykernel_14700\1588742831.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x)


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,hidden_dim = 256,head_num = 8):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.head_num = head_num
        self.q_layer = nn.Linear(hidden_dim,hidden_dim,bias=False)
        self.k_layer = nn.Linear(hidden_dim,hidden_dim,bias=False)
        self.v_layer = nn.Linear(hidden_dim,hidden_dim,bias=False)
        self.ff = nn.Linear(hidden_dim,hidden_dim)
        self.norm_layer = nn.LayerNorm(hidden_dim)
        self.attention_mask = torch.tril(torch.ones(batch,max_len,max_len))==0

        
    def forward(self,x):
        x,pad_mask = x
        q= self.q_layer(x).view(batch,max_len,self.head_num,self.hidden_dim//self.head_num).transpose(1,2)
        k = self.k_layer(x).view(batch,max_len,self.head_num,self.hidden_dim//self.head_num).transpose(1,2)
        v = self.v_layer(x).view(batch,max_len,self.head_num,self.hidden_dim//self.head_num).transpose(1,2)
        attention_score = torch.softmax(torch.masked_fill(torch.masked_fill(q@k.transpose(2,3),pad_mask.unsqueeze(1),-torch.inf),self.attention_mask.unsqueeze(1),-torch.inf),-1)
        attention_x = attention_score @v
        attention_x = attention_x.transpose(1,2).reshape((batch,max_len,self.hidden_dim))
        res_x = self.norm_layer(attention_x) + x
        ff_1 = self.ff(res_x)
        ff_2 = self.norm_layer(ff_1) + ff_1
        return (ff_2,pad_mask)
class TransformerDecoder(nn.Module):
    def __init__(self, hidden_dim = 256,head_num = 8,layers = 8,vocab_size =10000,max_len = 16,*args, **kwargs):
        super().__init__(*args, **kwargs)
        self.word_embedding = torch.nn.Embedding(vocab_size,hidden_dim)
        self.pos_embedding = torch.nn.Embedding(max_len,hidden_dim)
        self.attention_layers = nn.Sequential(*[MultiHeadAttention(hidden_dim,head_num)]*layers)
    def forward(self,x,pad_mask):
        embedding = word_embedding(x) + pos_embedding(torch.arange(max_len)).unsqueeze(0)
        out,_ = self.attention_layers((embedding,pad_mask))
        return out
class LLM(nn.Module):
     def __init__(self, hidden_dim = 256,head_num = 8,layers = 8,vocab_size =10000,max_len = 16):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.decoder = TransformerDecoder(hidden_dim ,head_num ,layers ,vocab_size,max_len)
        self.llm_head = nn.Sequential(nn.Linear(hidden_dim,vocab_size),nn.Softmax(dim=-1))

     def forward(self,x,y,pad_mask):
         decoder_out = self.decoder(x,pad_mask)
         out = self.llm_head(decoder_out)
         loss = nn.functional.cross_entropy(out.view(-1,self.vocab_size),y.view(-1))
         return loss
     def generate(self,x,max_new_tokens = 16):
         x = x + [encoding.eot_token]*(self.max_len-len(x)) if self.max_len-len(x)>0 else x
         x= torch.tensor(x)
         pad_mask = torch.ones(max_len,max_len)==0
         valid_length = len(x)
         pad_mask[:,length:] = True
         decoder_out = self.decoder(x,pad_mask)
         out = self.llm_head(decoder_out)
         return out
         
            

In [7]:
y.shape

torch.Size([2, 16])

In [8]:
import torch.optim as optim
model = LLM(vocab_size=vocab_size)
optimizer = optim.Adam(model.parameters(), lr=0.00001)
loss  = model(x,y,pad_mask)
loss.backward()
optimizer.step()